<a href="https://colab.research.google.com/github/Gunjannn07/FlyRank1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gunjannn07/FlyRank1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one pseudonymized client, one pseudonymized content item, and one report date. I will be using data from March 2026 (month = 2026-03).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- `gsc_impressions`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_engaged_sessions`
- `ga4_total_engagement_sec`

### Label / Proxy
- `gsc_clicks`

### Context
- `report_date`
- `client_hash_id`
- `content_hash_id`

### Excluded
- `gsc_sum_position` — similar with `gsc_avg_position` and not needed for this lane.
- `client_has_gsc` — indicates client-level availability rather than content performance.
- `client_has_ga4` — indicates client-level availability rather than content performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

print(ds)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


In [6]:
import duckdb
import os

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [os.environ["HF_TOKEN"]]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

query1 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 5
"""

con.sql(query1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────────┐
│ client_hash_id │ content_hash_id │ report_date │ row_count │
│    varchar     │     varchar     │    date     │   int64   │
├────────────────┴─────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

###### This proved the claim that there is no redundancy i.e 1 row= 1 client+ 1 content + 1 report date

In [7]:
#Row count and date range

query2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

con.sql(query2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

##### The March 2026 data consists of 9841378 rows with starting date as march 1st ending at march 31st.

In [8]:
#checking availability

query3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

con.sql(query3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘

##### Of the 9,841,378 March 2026 rows, 3,611,061 have GSC data available and 413,966 have GA4 data available. Availability was checked using `IS TRUE`.

### Five features

1. **`gsc_impressions`** — Available when `gsc_data_available IS TRUE`, because the value comes from GSC data for the report date.

2. **`gsc_avg_position`** — Available when `gsc_data_available IS TRUE`, because the value comes from GSC data for the report date.

3. **`ga4_pageviews`** — Available when `ga4_data_available IS TRUE`, because the value comes from GA4 data for the report date.

4. **`ga4_engaged_sessions`** — Available when `ga4_data_available IS TRUE`, because the value comes from GA4 data for the report date.

5. **`ga4_total_engagement_sec`** — Available when `ga4_data_available IS TRUE`, because the value comes from GA4 data for the report date.

In [9]:
feature_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_avg_position,
    ga4_pageviews,
    ga4_engaged_sessions,
    ga4_total_engagement_sec
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
LIMIT 100
"""

feature_frame = con.sql(feature_query).df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,ga4_total_engagement_sec
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,3.350000,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000000,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,4.928000,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,4.000000,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,2.272727,<NA>,<NA>,<NA>


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot provide a complete history for every client and content item. Data availability varies across rows, with GSC and GA4 data available for only part of the March 2026 slice. Therefore, missing data should not automatically be treated as zero.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.